In [1]:
from torch.utils.data import DataLoader, random_split
from utils.report import AverageMeter
from utils.metrics import calculate_accuracy

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

In [2]:
BATCH_SIZE = 256
LR = 0.001
ES_THRES = 5
SEED = 1234
OUTPUT_DIR = "./state_dicts/SVHN"

# Set Seed

In [3]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# Reference

- https://dacon.io/competitions/official/235614/codeshare/1300
- https://github.com/bentrevett/pytorch-image-classification/blob/master/2_lenet.ipynb

# Download Datasets

In [4]:
from torchvision.datasets import SVHN
import torchvision.transforms as transforms

In [5]:
download_root = '../datasets/SVHN_DATASET'
train_data = SVHN(download_root, split="train", download=True)
mean = train_data.data.mean() / 255
std = train_data.data.std() / 255

print(f'Calculated mean: {mean}')
print(f'Calculated std: {std}')

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Calculated mean: 0.45141874380092256
Calculated std: 0.19929124669110937


In [6]:
train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = [mean], std = [std])
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = [mean], std = [std])
])

In [7]:
train_valid_dataset = SVHN(download_root, transform=train_transforms, split="train", download=True)
train_dataset, valid_dataset = random_split(train_valid_dataset, [65931, 7326])
test_dataset = SVHN(download_root, transform=test_transforms, split="test", download=True)

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Using downloaded and verified file: ../datasets/SVHN_DATASET/test_32x32.mat


In [8]:
len(train_dataset)

65931

In [9]:
len(valid_dataset)

7326

# Models 

In [10]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [11]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

# Train/Eval Functions

In [12]:
def train_loop_classifier(feature_extractor, classifier, train_loader, loss_func, optimizer, 
                          summary_loss, summary_acc=None, device=None):
    feature_extractor.train()
    classifier.train()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        if device is not None:
            data, target = data.to(device), target.to(device)
            
        optimizer.zero_grad()
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output = classifier(feature)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        
        summary_loss.update(loss.detach().item(), BATCH_SIZE)
        if summary_acc is not None:
            acc = calculate_accuracy(output, target)
            summary_acc.update(acc, BATCH_SIZE)
        
    return summary_loss, summary_acc

def eval_loop_classifier(feature_extractor, classifier, valid_loader, loss_func, optimizer, 
                         summary_loss, summary_acc=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            if device is not None:
                data, target = data.to(device), target.to(device)

            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output = classifier(feature)

            loss = loss_func(output, target)

            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc is not None:
                acc = calculate_accuracy(output, target)
                summary_acc.update(acc, BATCH_SIZE)

    return summary_loss, summary_acc

# Training with Early Stopping

In [13]:
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()

    criterion = nn.CrossEntropyLoss()
    criterion.to(device)
    
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    train_loader = DataLoader(dataset=train_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    valid_loader = DataLoader(dataset=valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(True):
        epoch += 1
        summary_loss_train = AverageMeter()
        summary_acc_train = AverageMeter()
        summary_loss_valid = AverageMeter()
        summary_acc_valid = AverageMeter()

        summary_loss_train, summary_acc_train = train_loop_classifier(feature_extractor, classifier, train_loader, 
                                                   criterion, optimizer, summary_loss_train, summary_acc_train, device=device)
        summary_loss_valid, summary_acc_valid = eval_loop_classifier(feature_extractor, classifier, valid_loader, 
                                                   criterion, optimizer, summary_loss_valid, summary_acc_valid, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg} [train acc]{summary_acc_train.avg}  [valid loss]{summary_loss_valid.avg} [valid acc]{summary_acc_valid.avg} ")

        if best_loss is None:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch

        if best_loss > summary_loss_valid.avg:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch
            es_count = 0
        else:
            es_count += 1

        if es_count == ES_THRES:
            break

    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [14]:
best_epoch = run_training()

[epoch]1 [train loss]1.321092888828396 [train acc]0.5641935467720032  [valid loss]0.7660385431914494 [valid acc]0.7791527509689331 
[epoch]2 [train loss]0.610239718196004 [train acc]0.8250101804733276  [valid loss]0.5690068623115276 [valid acc]0.8418982625007629 
[epoch]3 [train loss]0.4941899197739224 [train acc]0.8601400852203369  [valid loss]0.4988347877716196 [valid acc]0.8597296476364136 
[epoch]4 [train loss]0.43432635606028314 [train acc]0.8758686780929565  [valid loss]0.4545889075460105 [valid acc]0.8716291189193726 
[epoch]5 [train loss]0.3932596272391866 [train acc]0.8871889710426331  [valid loss]0.42124662830911835 [valid acc]0.8811227679252625 
[epoch]6 [train loss]0.3603572742652523 [train acc]0.8953449130058289  [valid loss]0.416460036203779 [valid acc]0.8822003602981567 
[epoch]7 [train loss]0.3371871412377949 [train acc]0.9026553630828857  [valid loss]0.4068547374215619 [valid acc]0.8854655623435974 
[epoch]8 [train loss]0.3155542718925217 [train acc]0.9082136154174805 

# Second Training

In [15]:
def run_second_training(best_epoch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()
    
    train_valid_loader = DataLoader(dataset=train_valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    criterion = nn.CrossEntropyLoss()
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    epoch = 0
    for _ in range(best_epoch):
        epoch += 1
        summary_loss_train = AverageMeter()
    #     summary_acc_train = AverageMeter()

        summary_loss_train, _ = train_loop_classifier(feature_extractor, classifier, train_valid_loader, 
                                                   criterion, optimizer, summary_loss_train, None, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg}")
    return feature_extractor, classifier

In [16]:
feature_extractor, classifier = run_second_training(best_epoch)

[epoch]1 [train loss]1.2662767450153205
[epoch]2 [train loss]0.5938292582899021
[epoch]3 [train loss]0.4976216411341358
[epoch]4 [train loss]0.44014043810060216
[epoch]5 [train loss]0.403633415439403
[epoch]6 [train loss]0.37113146608507175
[epoch]7 [train loss]0.34505137982891826
[epoch]8 [train loss]0.325498485502881
[epoch]9 [train loss]0.305587297584537
[epoch]10 [train loss]0.29211695783021974
[epoch]11 [train loss]0.27793887361416836
[epoch]12 [train loss]0.2624793123285114


In [17]:
def run_test(feature_extractor, classifier):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(feature_extractor, classifier, test_loader, 
                                               criterion, None, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [18]:
run_test(feature_extractor, classifier)

[test loss]0.43791083231860517 [test acc]0.8849083781242371


In [21]:
torch.save(feature_extractor.state_dict(), os.path.join(OUTPUT_DIR, "feature_extractor.pth"))
torch.save(classifier.state_dict(), os.path.join(OUTPUT_DIR, "classifier.pth"))